# 05_Hyperparameter_Tuning.ipynb

Hyperparameter tuning helps us find the best combination of parameters that gives the highest performance on unseen data.

Instead of manually trying different values of `C`, `gamma`, or `kernel`, we automate the search.

---

# Workflow

```text
Load Dataset
      ↓
Train-Test Split
      ↓
Scaling
      ↓
Define Parameter Grid
      ↓
Cross Validation
      ↓
GridSearchCV / RandomizedSearchCV
      ↓
Best Parameters
      ↓
Train Best Model
      ↓
Evaluate
```

---

# 1. Why Hyperparameter Tuning?

Suppose we train

```python
model = SVC(
    C=1,
    gamma="scale"
)
```

How do we know that

* `C=1` is the best?
* `gamma="scale"` is the best?
* `kernel="rbf"` is the best?

We don't.

Hyperparameter tuning searches different combinations automatically.

---

# 2. Cross Validation

Instead of evaluating on a single train-test split,

Cross Validation trains multiple models.

Example:

```text
Dataset

Fold 1
Train → Test

Fold 2
Train → Test

Fold 3
Train → Test

Fold 4
Train → Test

Fold 5
Train → Test

↓

Average Accuracy
```

This provides a more reliable estimate of model performance.

---

# 3. GridSearchCV

Grid Search tries **every possible combination** of the specified parameters.

---

## Import

```python
from sklearn.model_selection import GridSearchCV
```

---

## Example Parameter Grid

```python
param_grid = {
    "C": [0.1, 1, 10],
    "gamma": [0.01, 0.1, 1],
    "kernel": ["linear", "rbf"]
}
```

Number of combinations

```text
3 × 3 × 2 = 18
```

Every one of these 18 models is trained and evaluated.

---

## Create Grid Search

```python
grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)
```

---

## Train

```python
grid.fit(X_train, y_train)
```

---

## Best Parameters

```python
print(grid.best_params_)
```

Example

```text
{
'C':10,
'gamma':0.1,
'kernel':'rbf'
}
```

---

## Best Score

```python
print(grid.best_score_)
```

Example

```text
0.982
```

---

## Best Model

```python
best_model = grid.best_estimator_
```

No need to create another SVC manually.

---

# 4. RandomizedSearchCV

Instead of checking **every combination**,

Randomized Search checks only a random subset.

---

## Import

```python
from sklearn.model_selection import RandomizedSearchCV
```

---

## Parameter Distribution

```python
param_dist = {
    "C": [0.01,0.1,1,10,100],
    "gamma": [0.001,0.01,0.1,1],
    "kernel": ["linear","rbf","poly"]
}
```

---

## Create Random Search

```python
random = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    random_state=42
)
```

---

## Train

```python
random.fit(X_train,y_train)
```

---

## Best Parameters

```python
print(random.best_params_)
```

---

# 5. Grid Search vs Random Search

| Feature                         | Grid Search | Random Search   |
| ------------------------------- | ----------- | --------------- |
| Checks Every Combination        | ✅           | ❌               |
| Faster                          | ❌           | ✅               |
| Finds True Best in Search Space | ✅           | ❌ (Approximate) |
| Good for Small Search Space     | ✅           | ❌               |
| Good for Large Search Space     | ❌           | ✅               |

---

# 6. Choosing Parameter Ranges

## `C`

Common values

```python
[0.01,0.1,1,10,100]
```

---

## `gamma`

Common values

```python
[0.001,0.01,0.1,1]
```

---

## Polynomial Degree

```python
[2,3,4,5]
```

---

## Kernel

```python
["linear","rbf","poly","sigmoid"]
```

Usually,

* start with `"rbf"`
* compare with `"linear"`

---

# 7. Best Practices

* Scale features before tuning.
* Use Cross Validation.
* Start with a small search space.
* Use `RandomizedSearchCV` for very large parameter spaces.
* Use `GridSearchCV` when the search space is small.

---

# 8. Complete Example

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Load data
X, y = load_breast_cancer(return_X_y=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Parameter Grid
param_grid = {
    "C":[0.1,1,10],
    "gamma":[0.01,0.1,1],
    "kernel":["linear","rbf"]
}

# Grid Search
grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train,y_train)

print("Best Parameters")
print(grid.best_params_)

print("Best Score")
print(grid.best_score_)

best_model = grid.best_estimator_

print(best_model.score(X_test,y_test))
```

---

# Common Mistakes

| Mistake                           | Why It Happens                                                     |
| --------------------------------- | ------------------------------------------------------------------ |
| Large parameter grid              | Training becomes very slow because every combination is evaluated. |
| Forgetting feature scaling        | SVM performance can degrade significantly.                         |
| Tuning on the test set            | Leads to data leakage and overly optimistic results.               |
| Very large `C` and `gamma` ranges | Increases the chance of overfitting and unnecessary computation.   |

---
